In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, zipfile
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix
from baseline import calculate_resilience_cost
import matplotlib.pyplot as plt
SEED = 41
np.random.seed(SEED)

# --- Load data ---
df = pd.read_csv("data/train.csv", index_col=0)
test_df = pd.read_csv("data/test.csv",  index_col=0)
cost_matrix_df = pd.read_csv("data/cost_matrix.csv", index_col=0)

display(df)

In [ ]:
display(cost_matrix_df.head())


In [ ]:

# Makes no sense to do it this way, ruins the ordinal structure, let's set it to red->orange->yellow->green
cost_matrix_df = cost_matrix_df.sort_values(by=["0"])
cost_matrix_df = cost_matrix_df[['0', '3', '1', '2']]
cost_matrix = np.flip(cost_matrix_df.values)
display(cost_matrix)

In [ ]:
# Data inspection
display(df.describe())
df.info()


In [ ]:
# Let's change the types for efficiency (maybe it happens already automatically though..)


# is_magnitude_int = all(x.is_integer() for x in df["magnitude"])
# print("Is magnitude actually ints: ", is_magnitude_int)
# # Magnitude is proper float

# is_depth_int = all(x.is_integer() for x in df["depth"])
# print("Is depth actually ints: ", is_depth_int)
# # depth should be uint
# df.depth = df.depth.astype("UInt16")

# is_cdi_int = all(x.is_integer() for x in df["cdi"])
# print("Is cdi actually ints: ", is_cdi_int)
# # cdi should be uint
# df.cdi = df.cdi.astype("UInt8")

# is_mmi_int = all(x.is_integer() for x in df["mmi"])
# print("Is mmi actually ints: ", is_mmi_int)
# # mmi should be int
# df.mmi = df.mmi.astype("UInt8")

# is_sig_int = all(x.is_integer() for x in df["sig"])
# print("Is sig actually ints: ", is_sig_int)
# # sig should be int
# df.sig = df.sig.astype("Int16")

# This actually didn't do anything (also should be applied to test df anyway), it probably gets converted back to floats for computation anyway...

In [ ]:
df.info()

In [ ]:
df.alert = pd.Categorical(df.alert, ["green", "yellow", "orange", "red"], ordered=True)
df.sort_values(by=["alert"], inplace=True)
display(df.alert)
df.reset_index(drop=True, inplace=True)
len(df) - len(df.drop_duplicates())
df.info()
# No dups

In [ ]:

sns.barplot(data=df["alert"], estimator="size") 

In [ ]:

# sns.histplot(data=df, x = "magnitude", hue="alert",)
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="magnitude")


In [ ]:
sns.scatterplot(data=df, x=df.magnitude, y=df.index, hue=df.alert)

In [ ]:
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="depth")

In [ ]:
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)
# Some outliers for orange


In [ ]:
# outliers = (df.alert == "orange") & (df.depth > 100)
# df.drop(df[outliers].index, inplace=True)
print("After drops")
sns.scatterplot(data=df, x=df.depth, y=df.index, hue=df.alert)


In [ ]:
sns.histplot(data=df["cdi"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="cdi")


In [ ]:
plt.figure(figsize = (8, 10))

sns.scatterplot(data=df, x=df.cdi, y=df.index, hue=df.alert)


In [ ]:
sns.histplot(data=df["mmi"])

sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="mmi")


In [ ]:
plt.figure(figsize = (10, 8))

sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)
# Looks like some outliers for everything except red

In [ ]:
# outliers = ((df.alert == "orange") & (df.mmi == 9)) | ((df.alert == "yellow") & (df.mmi == 9)) | ((df.alert == "green") & (df.mmi < 3)) # This one seems tricky on the performance
# df.drop(df[outliers].index, inplace=True)
sns.scatterplot(data=df, x=df.mmi, y=df.index, hue=df.alert)


In [ ]:

sns.histplot(data=df["sig"])
sns.FacetGrid(df, row="alert").map_dataframe(sns.histplot, x="sig")

In [ ]:
plt.figure(figsize = (10, 8))
sns.scatterplot(data=df, x=df.sig, y=df.index, hue=df.alert)


In [ ]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder ,OrdinalEncoder
from sklearn.metrics import make_scorer

categories=[["green", "yellow", "orange", "red"]]
color_to_int = {"green": 3, "yellow": 2, "orange": 1, "red": 0}
# --- Encode target and split ---
# enc = OrdinalEncoder(categories=categories, dtype=int, )
# enc = LabelEncoder()
print("Enc: ")
# display(enc)
# enc.fit(df.loc[:, ["alert"]])
print("Pre mapping y:")
display(df.alert)
y = df.alert.map(color_to_int).astype(int)
print("Post mapping y:")
display(y)
X = df.drop("alert", axis=1)
X_final = test_df
display(X.corr())
display(df.info())

int_to_color = {3: "green", 2: "yellow", 1: "orange", 0: "red"}
def mean_resilience_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    rci = calculate_resilience_cost(cm, cost_matrix)
    return rci / len(y_true)

# We use resilience_score rather than accuracy, since this is what will be competed upon
resilience_scorer = make_scorer(mean_resilience_score, greater_is_better=False, )

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import PowerTransformer

# pipe = make_pipeline(StandardScaler())
# numeric_features = ["magnitude", "depth", "cdi", "mmi", "sig"]
# numeric_transformer = Pipeline(
#     steps=[("scaler", StandardScaler())]
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, numeric_features),
#         # ("cat", categorical_transformer, categorical_features),
#     ]
# )

# numeric_log = ['depth'] # Strongly skewed to the right
numeric_scale_only = [ 'depth', 'magnitude', 'cdi', 'mmi', 'sig']

preprocessor = ColumnTransformer(
    transformers=[
        # ('power', PowerTransformer(method='yeo-johnson'), numeric_log),
        ('scale', StandardScaler(), numeric_scale_only),
    ]
)




In [ ]:
X.describe()

In [ ]:

X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=SEED, stratify=y_val_test
)
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

In [ ]:
display(pd.DataFrame(X_train).describe())

## SVM

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.compose import TransformedTargetRegressor
def make_svm(probability=False):
    class_weight = {
            0: 0.075,
            1: 0.35,
            2: 0.5,
            3: 0.125,
        }
    class_weight = {
        0: 0.6,
        1: 0.30,
        2: 0.075,
        3: 0.025,
    }
    # clf = Pipeline(
    #     steps=[("preprocessor", preprocessor), ("classifier", SVC(C=700, class_weight=class_weight, probability=probability))]
    #     # steps=[("classifier", SVC(C=700, class_weight=class_weight))]
    #     # steps=[("preprocessor", preprocessor), ("classifier", LinearRegression())]
    # )
    return SVC(C=700, class_weight=class_weight, probability=probability)

In [ ]:
from matplotlib import pyplot as plt
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold





# The class weights are very interesting, they seem to make a big impact since recall can be slightly controlled 



outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

param_grid = {
    'classifier__gamma': ["scale"],
    'classifier__kernel': [ "rbf"],
    'classifier__C': np.arange(10, 100, 2)
    }
    # 'classifier_class_weigths_': [{"}
    # 'classifier__C': np.linspace(1, 1)}

grid_search = GridSearchCV(
    estimator=make_svm(),
    param_grid=param_grid,
    scoring=resilience_scorer,
    cv=inner_cv)

# nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring=resilience_scorer, )


In [ ]:

print(f"Outer CV Scores (Generalization Error): {nested_scores}")
print(f"Mean Nested Stratified CV Resilience Estimate: {np.mean(nested_scores):.4f}")

# Plotting the nested CV scores
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(nested_scores) + 1), nested_scores, color='purple')
plt.axhline(np.mean(nested_scores), color='red', linestyle='--', label=f'Mean Nested Resilience Score ({np.mean(nested_scores):.4f})')
plt.title('Nested Stratified CV Scores (Tuning Error)')
plt.xlabel('Outer Fold Number')
plt.ylabel('Resilience Score')
plt.ylim(-0, -50)
plt.legend()
plt.show()


In [ ]:
# from sklearn.metrics import ConfusionMatrixDisplay

# grid_search.fit(np.concat([X_train, X_val]), np.concat([y_train, y_val]))
# display(grid_search.best_params_)
# display(grid_search.best_score_)
# clf = grid_search

# # --- Validate ---
# y_test_pred = clf.predict(X_test)
# f1 = f1_score(y_test, y_test_pred, average="macro")
# cm = confusion_matrix(y_test, y_test_pred)
# rci = calculate_resilience_cost(cm, cost_matrix)

# print(classification_report(y_test, y_test_pred))

# cm_display = ConfusionMatrixDisplay(cm, display_labels=["green", "yellow", "orange", "red"]).plot()
# print(f"F1 (macro): {f1:.3f}")
# # print("Confusion matrix:\n", cm)
# print(f"Mean Resilience Cost: {rci / len(y_test):.2f}")
# print(f"Coda Resilience Cost: {rci / len(y_test) * 450:.2f}")

# GradientBoosting

In [ ]:

# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.preprocessing import PowerTransformer

# numeric_log = ['depth'] # Strongly skewed to the right
# numeric_scale_only = ['magnitude', 'cdi', 'mmi', 'sig']

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('power', PowerTransformer(method='yeo-johnson'), numeric_log),
#         ('scale', StandardScaler(), numeric_scale_only),
#     ]
# )

# clf_boosting = Pipeline(
#     steps=[("preprocessor", preprocessor), ("classifier", GradientBoostingClassifier())]
# )

# param_grid = {
#     "classifier__n_estimators": [150, 300],
#     "classifier__learning_rate": [0.03, 0.1],
#     "classifier__max_depth": [2, 3]
#     }

# mod_gbc = GridSearchCV(
#     estimator=clf_boosting,
#     param_grid=param_grid,
#     scoring=resilience_scorer,
#     cv=inner_cv)

# nested_scores = cross_val_score(mod_gbc, X, y, cv=outer_cv, scoring="accuracy" )


## Neural Network

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from keras.layers import Activation, Dropout, Flatten, Dense
import keras.regularizers
from tensorflow.keras.callbacks import EarlyStopping
from keras import regularizers, Input
from keras.losses import sparse_categorical_crossentropy
from keras.utils import set_random_seed
import tensorflow as tf
import coral_ordinal as coral
from scipy import special

# Used an interesting ordinal classification method from: 
# Cao, W., Mirjalili, V., & Raschka, S. (2019). Rank-consistent ordinal regression for neural networks. arXiv preprint arXiv:1901.07884, 6.

set_random_seed(SEED)
cost_matrix_tensor = tf.constant(cost_matrix, dtype=tf.float32)


def cost_sensitive_loss(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true), tf.int32)
    row = tf.gather(cost_matrix_tensor, y_true)
    # y_pred must be from softmax
    return tf.reduce_mean(tf.reduce_sum(row * y_pred, axis=1))


def hybrid_loss(alpha=0.5):
    def loss(y_true, y_pred):
        ce = sparse_categorical_crossentropy(y_true, y_pred)
        cs = cost_sensitive_loss(y_true, y_pred)
        return alpha * ce + (1 - alpha) * cs

    return loss


class CustomOrdinalClassification():
    def __init__(self, num_features, num_classes=None):
        regularization_strength = 0.0001
        self.model = Sequential(
            [
                # Input(shape = (X_train.shape[1], )),
                Dense(512, 
                    input_shape=(num_features,),
                    activation="elu",
                    kernel_regularizer=regularizers.l2(regularization_strength),
                ),
                Dense(512, kernel_regularizer=regularizers.l2(regularization_strength), activation="elu"),
                Dropout(0.2),
                Dense(512, kernel_regularizer=regularizers.l2(regularization_strength), activation="elu"),
                Dropout(0.2),
                # Dense(1024, activation='relu'),
                # Dense(1024, activation='relu'),
                # Dense(1024, activation='relu'),
                # Dense(1024, activation='relu'),
                # Dense(1024, activation='relu'),
                # Dropout(0.3),
                # Dense(32, activation="relu"),
                # Dense(32, activation='relu'),
                coral.CoralOrdinal(num_classes = 4),
                # Dense(4, activation="softmax"),
            ]
        )

    
        # class_weight = {
        #     0: 0.025,
        #     1: 0.075,
        #     2: 0.30,
        #     3: 0.6,
        # }
        weights = tf.constant([1, 0.3, 0.1])
        # weights = tf.constant([1, 1, 1])
        # Maybe change adam params
        self.model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.05), loss = coral.OrdinalCrossEntropy(num_classes=4, importance_weights=weights),
                metrics = [coral.MeanAbsoluteErrorLabels()])
        
    
    def predict_cum_proba(self, X_pred):
        # Note that these are ordinal (cumulative) logits, not probabilities or regular logits.
        ordinal_logits = self.model.predict(X_pred)
        
        # probs = coral.ordinal_softmax(ordinal_logits).numpy()


        # Convert from logits to label probabilities. This is initially a tensorflow tensor.
        # tensor_probs = coral.ordinal_softmax(ordinal_logits)
        # probs = tensor_probs.numpy()
        return pd.DataFrame(ordinal_logits).apply(special.expit)

        
    def predict(self, X_pred):
        # probs_df = pd.DataFrame(self.predict_proba(X_pred).numpy())
        cum_probs = self.predict_cum_proba(X_pred)

        y_test_pred = cum_probs.apply(lambda x: x > 0.5).sum(axis = 1)
        return  y_test_pred
    
mod_coral = CustomOrdinalClassification(X.shape[1], num_classes=4)

early_stopping_monitor = EarlyStopping(
    patience=30, min_delta=0.000, restore_best_weights=True, verbose=True
)
early_stopping_monitor = EarlyStopping(
    patience=200, restore_best_weights=True, verbose=True
)

history = mod_coral.model.fit(# np.concat([X_train, X_val]),
    # np.concat([y_train , y_val]),
    # validation_data=(X_test,y_test),
    np.concat([X_train]),
    np.concat([y_train]),
    validation_data=(X_val, y_val),
    shuffle=True,
    batch_size=16384 * 1024 * 2, # cursed
    # batch_size=64,
    epochs=2000,
    verbose=1,
    callbacks=[early_stopping_monitor],
    # class_weight=class_weight,
)



# mod_ann = CustomOrdinalClassification(5,4).model
mod_coral.model.summary()

# weights = tf.constant([0.1, 0.2,0.4])
# mod_ann.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.05),
#               loss = coral.OrdinalCrossEntropy(num_classes = 4, importance_weights=weights),
#               metrics = [coral.MeanAbsoluteErrorLabels()])



In [ ]:
# print(mod_ann.get.get_metrics_result())
# The output layer is the last layer in the model.
last_layer = len(mod_coral.model.layers) - 1
print(mod_coral.model.get_weights()[0])
# Tune weights to controll recall
# mod_ann.layers[last_layer].set_weights(weights = [mod_ann.layers[last_layer].get_weights()[0], np.array([  7  ,  -1, -10.0])])
# mod_ann.layers[last_layer].set_weights(weights = [mod_ann.layers[last_layer].get_weights()[0], np.array([  1.5, -2.5, -7.3533816])])
# mod_ann.layers[last_layer].set_weights(weights = [mod_ann.layers[last_layer].get_weights()[0], np.array([  3.5, -2.5, -7])])



# probs_df.head()

In [ ]:
# Probs to labels
# labels = probs_df.idxmax(axis = 1)
# labels.values


In [ ]:
# mod_ann.save("ann_5.keras")

In [ ]:
# extract the loss and metric values
loss = history.history['loss']
acc = history.history['mean_absolute_error_labels']
val_loss = history.history['val_loss']
val_acc = history.history['val_mean_absolute_error_labels']

# plot the loss
plt.plot(loss, label='loss')
plt.plot(val_loss, label='val_loss')
plt.legend()
plt.show()

# plot the accuracy
plt.plot(acc, label='mean_absolute_error_labels')
plt.plot(val_acc, label='val_mean_absolute_error_labels')
plt.legend()
plt.show()

In [ ]:

# meta_features = []
# meta_targets = []

# X_train_full, X_test, y_train_full, y_test = train_test_split(
#     X, y,
#     test_size=0.2,
#     random_state=SEED,
#     stratify=y
# )

# kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# for train_idx, val_idx in kf.split(X_train_full, y_train_full):
#     X_tr_raw = X_train_full.iloc[train_idx]
#     X_val_raw = X_train_full.iloc[val_idx]

#     y_tr = y_train_full.iloc[train_idx]
#     y_val = y_train_full.iloc[val_idx]

#     # Preprocessing per fold 
#     scaler = StandardScaler().fit(X_tr_raw)
#     X_tr = scaler.transform(X_tr_raw)
#     X_val = scaler.transform(X_val_raw)

#     # Level-0 SVM 
#     svc = make_svm(probability=True)
#     svc.fit(X_tr, y_tr)
#     svc_val_pred = svc.predict_proba(X_val)

#     # Level-0 Keras 
#     nn = CustomOrdinalClassification(X.shape[1], 4)
#     nn.model.fit(X_tr, y_tr, epochs=400, batch_size=64, verbose=0, callbacks=[early_stopping_monitor])
#     nn_val_pred = nn.predict_proba(X_val)

#     # Add stacked features 
#     meta_features.append(np.hstack([svc_val_pred, nn_val_pred]))
#     meta_targets.append(y_val)


In [ ]:
# meta_X = np.vstack(meta_features)
# meta_y = np.hstack(meta_targets)
# meta_y_oh = keras.utils.to_categorical(meta_y, num_classes=4)
# print(meta_X.shape, meta_y.shape, meta_y_oh.shape)

In [ ]:

# full_scaler = StandardScaler().fit(X_train_full)
# X_train_scaled = full_scaler.transform(X_train_full)

# svc_full = make_svm(probability=True).fit(X_train_scaled, y_train_full)

# nn_full = CustomOrdinalClassification(X.shape[1], 4)
# nn_full.model.fit(X_train_scaled, y_train_full, epochs=200, batch_size=64)


In [ ]:
# svc_pred_full = svc_full.predict_proba(X_train_scaled)
# nn_pred_full = nn_full.predict_proba(X_train_scaled)

# final_meta_X = np.hstack([svc_pred_full, nn_pred_full])
# probs = nn.predict_proba(X_val)
# print("Row sums:", probs.sum(axis=1)[:10])
# print("Min probability:", probs.min())
# print("Max probability:", probs.max())


In [ ]:
# df_check = pd.DataFrame({
#     "true": meta_y,
#     "svc_pred_class": np.argmax(meta_X[:, :4], axis=1),
#     "nn_pred_class": np.argmax(meta_X[:, 4:], axis=1),
# })

# print(df_check.head(20))
# print(df_check.sample(20))



In [ ]:
# df_check = pd.DataFrame({
#     "true": meta_y,
#     "svc_pred_class": np.argmax(meta_X[:, :4], axis=1),
#     "nn_pred_class": np.argmax(meta_X[:, 4:], axis=1),
# })

# print(df_check.head(20))
# print(df_check.sample(20))



In [ ]:
# print(meta_X[:10])
# print(meta_y[:10])
# print(np.argmax(meta_X[:10,:4], axis=1))
# print(np.argmax(meta_X[:10,4:], axis=1))


In [ ]:
# def cost_matrix_loss(y_true, y_pred):
#     # y_true: (batch, 4) one-hot
#     # y_pred: (batch, 4) softmax
#     cost_per_pred = tf.matmul(y_pred, cost_matrix_tensor, transpose_b=True)
#     # expected cost given y_true one-hot
#     return tf.reduce_mean(tf.reduce_sum(y_true * cost_per_pred, axis=1))

# def make_level1_keras(input_dim):
#     model = keras.Sequential([
#         keras.layers.Input(shape=(input_dim,)),  
#         keras.layers.Dense(16, activation="relu"),
#         keras.layers.Dense(4, activation="softmax")
#     ])

#     model.compile(
#         optimizer=keras.optimizers.Adam(0.005),
#         # loss=cost_matrix_loss,
#         # metrics=[cost_matrix_loss, "accuracy"]
#         loss="categorical_crossentropy",
#         metrics=["accuracy"]
#     )
#     return model

# meta_model = make_level1_keras(final_meta_X.shape[1])
# history = meta_model.fit(final_meta_X, meta_y_oh,
#                epochs=1000, batch_size=64, verbose=1)


In [ ]:
# from sklearn.metrics import confusion_matrix

# def stacked_predict_proba(X_new):
#     X_new_scaled = full_scaler.transform(X_new)
#     p_svc = svc_full.predict_proba(X_new_scaled)
#     p_nn  = nn_full.predict_proba(X_new_scaled)
#     stacked = np.hstack([p_svc, p_nn])
#     return meta_model.predict(stacked, verbose=0)

# def stacked_predict(X_new):
#     p = stacked_predict_proba(X_new)
#     return np.argmax(p, axis=1)

# # Predictions on test
# y_test_pred = stacked_predict(X_test)



In [ ]:
# loss = history.history['cost_matrix_loss']
# acc = history.history['accuracy']
# val_loss = history.history['val_cost_matrix_loss']
# val_acc = history.history['val_accuracy']

# # plot the loss
# plt.plot(loss, label='cost_matrix_loss')
# plt.plot(val_loss, label='val_cost_matrix_loss')
# plt.legend()
# plt.show()

# # plot the accuracy
# plt.plot(acc, label='accuracy')
# plt.plot(val_acc, label='val_accuracy')
# plt.legend()
# plt.show()

In [ ]:
y_test_pred = mod_coral.predict(X_test)
print(max(y_test_pred))
# print(cm)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

# y_test_pred_prob = mod_ann.predict(X_test)
# y_test_pred = y_test_pred_prob.argmax(axis=1)




# Check bias terms: these should be in descending order.
display(mod_coral.model.layers[last_layer].get_weights()[1])

plt.plot(mod_coral.model.layers[last_layer].get_weights()[1])
plt.show()
# temp = mod_ann.layers[last_layer].get_weights()[1]
#  3.4974706 -1.0368032 -7.3533816]
# print(temp)

# old_weights 
y_test_pred = mod_coral.predict(X_test)
f1 = f1_score(y_test, y_test_pred, average="macro")
cm = confusion_matrix(y_test, y_test_pred)
rci = calculate_resilience_cost(cm, cost_matrix)

print(classification_report(y_test, y_test_pred))

cm_display = ConfusionMatrixDisplay(cm, display_labels=["red",    "orange", "yellow", "green"]).plot()
print(f"F1 (macro): {f1:.3f}")
# print("Confusion matrix:\n", cm)
print(f"Mean Resilience Cost: {rci / len(y_test):.2f}")
print(f"Coda Resilience Cost: {rci / len(y_test) * 450:.2f}")

In [ ]:
# print(f"Outer CV Scores (Generalization Error): {nested_scores}")
# print(f"Mean Nested Stratified CV Resilience Estimate: {np.mean(nested_scores):.4f}")

# # Plotting the nested CV scores
# plt.figure(figsize=(8, 5))
# plt.bar(range(1, len(nested_scores) + 1), nested_scores, color='purple')
# plt.axhline(np.mean(nested_scores), color='red', linestyle='--', label=f'Mean Nested Resilience Score ({np.mean(nested_scores):.4f})')
# plt.title('Nested Stratified CV Scores (Tuning Error)')
# plt.xlabel('Outer Fold Number')
# plt.ylabel('Resilience Score')
# plt.ylim(-0, 1)
# plt.legend()
# plt.show()



In [ ]:

# # # --- Train baseline model ---
# clf = KNeighborsClassifier(n_neighbors=10)
# clf.fit(X_train, y_train)
# clf


In [ ]:
from baseline import create_submission

# best_params = grid_search.best_params_
# final_clf = Pipeline(
#     steps=[
#         ("preprocessor", preprocessor),
#         ("classifier", SVC(
#             C=best_params["classifier__C"],
#             gamma=best_params["classifier__gamma"],
#             kernel=best_params["classifier__kernel"],
#             class_weight=class_weight
#         ))
#     ]
# )

# final_clf.fit(X, y)

# For Coral model
ordinal_logits = mod_ann.predict(preprocessor.transform(X_final))
y_final = pd.DataFrame(ordinal_logits).apply(special.expit).apply(lambda x: x > 0.5).sum(axis = 1)

# y_final = mod_ann.predict(preprocessor.transform(X_final)).argmax(axis=1)
# display(best_params)
display([int_to_color[i] for i in y_final][1:3])
create_submission([int_to_color[i] for i in y_final])